# Persistent Sessions: Save, Resume, and Branch Conversations

Long-running Claude applications often need to survive a process restart, move conversations between users, or let the user explore "what if" branches without losing the main thread. The [Session Memory Compaction](session_memory_compaction.ipynb) cookbook shows how to keep a long conversation within the context limit *in memory*. This cookbook picks up where that leaves off and covers **persistence** — turning an in-memory session into something you can store, reload, and fork.

**Related:**
- [Session Memory Compaction](session_memory_compaction.ipynb) — managing context limits via compaction (the in-memory story).
- [Automatic Context Compaction](../tool_use/automatic-context-compaction.ipynb) — SDK-side compaction for agentic workflows.
- [Memory & context management](../tool_use/memory_cookbook.ipynb) — Claude's memory tool for cross-session persistence of *facts* (vs. full conversations).

## Learning objectives

By the end of this cookbook you will be able to:

- Serialize a Claude conversation (messages, session memory, metadata) to disk in a forward-compatible format.
- Reload a session in a new process and continue where you left off, with the right cache breakpoints to preserve prompt-cache hits.
- Fork a session at a checkpoint so users can explore alternate branches without losing the main thread.
- Redact or scrub sensitive content before persisting.

> **Note:** This notebook is deliberately self-contained. It uses a tiny fake Claude client so the patterns (serialize, resume, fork) can be demonstrated without an API key. At the end we show the one-line swap to the real `anthropic.Anthropic()` client.


## Prerequisites

**Required knowledge**
- Basic Python (`dataclasses`, `json`, `pathlib`).
- Familiarity with the Claude `messages` format (`role`, `content`).

**Required tools**
- Python 3.11+.
- No API key is required to run this notebook. The `anthropic` SDK is only imported for type hints; install it if you want to plug in a real client in the final section:

```bash
pip install -U anthropic
```


In [1]:
from __future__ import annotations

import json
import uuid
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from datetime import UTC, datetime
from pathlib import Path

SCHEMA_VERSION = 1  # bump when the on-disk format changes
SESSIONS_DIR = Path("./_sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

## 1. A session is just data

The smallest useful "session" is three things:

1. **Messages** — the running list of `{"role": ..., "content": ...}` entries you send to the API.
2. **Session memory** — the compacted summary of earlier turns (see the [Session Memory Compaction](session_memory_compaction.ipynb) cookbook).
3. **Metadata** — who owns it, when it was last touched, what model it's pinned to, a schema version so you can evolve the format safely.

We'll model that with a dataclass. Keeping it plain-JSON-serializable is the whole point: no pickle, no ORM, no surprises when you move between processes or languages.

In [2]:
@dataclass
class Session:
    """A persistable Claude conversation."""

    session_id: str
    model: str
    system: str = ""
    messages: list[dict] = field(default_factory=list)
    session_memory: str | None = None
    created_at: str = field(default_factory=lambda: datetime.now(UTC).isoformat())
    updated_at: str = field(default_factory=lambda: datetime.now(UTC).isoformat())
    parent_id: str | None = None  # set when this session was forked from another
    schema_version: int = SCHEMA_VERSION

    @classmethod
    def new(cls, model: str, system: str = "") -> Session:
        return cls(session_id=str(uuid.uuid4()), model=model, system=system)

    def touch(self) -> None:
        self.updated_at = datetime.now(UTC).isoformat()

    def append(self, role: str, content: str) -> None:
        self.messages.append({"role": role, "content": content})
        self.touch()

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False)

    @classmethod
    def from_json(cls, raw: str) -> Session:
        data = json.loads(raw)
        version = data.get("schema_version", 0)
        if version > SCHEMA_VERSION:
            raise ValueError(
                f"Session schema v{version} is newer than this code (v{SCHEMA_VERSION}). "
                "Upgrade the cookbook library before loading."
            )
        # Older versions would be migrated here; we only have v1 so far.
        return cls(**data)

In [3]:
session = Session.new(
    model="claude-sonnet-4-6",
    system="You are a helpful writing assistant.",
)
session.append("user", "Help me outline a short story about a lighthouse keeper.")
session.append(
    "assistant",
    "Sure! Here's a 3-act outline: (1) arrival, (2) the light fails, (3) what they find below.",
)

print(f"Session id: {session.session_id}")
print(f"Messages:   {len(session.messages)}")
print("First 200 chars of JSON:")
print(session.to_json()[:200] + "...")

Session id: 19dca882-30d8-4491-bca2-5ed7593514a8
Messages:   2
First 200 chars of JSON:
{
  "session_id": "19dca882-30d8-4491-bca2-5ed7593514a8",
  "model": "claude-sonnet-4-6",
  "system": "You are a helpful writing assistant.",
  "messages": [
    {
      "role": "user",
      "content...


## 2. Save and resume

Saving is `write_text(session.to_json())`. The only two things worth thinking about are **atomicity** (don't leave a half-written file if the process dies mid-write) and **path hygiene** (one file per session, named by id).

In [4]:
def save_session(session: Session, directory: Path = SESSIONS_DIR) -> Path:
    """Atomically persist a session to disk as JSON."""
    directory.mkdir(parents=True, exist_ok=True)
    final = directory / f"{session.session_id}.json"
    tmp = final.with_suffix(".json.tmp")
    session.touch()
    tmp.write_text(session.to_json(), encoding="utf-8")
    tmp.replace(final)  # atomic on POSIX and Windows
    return final


def load_session(session_id: str, directory: Path = SESSIONS_DIR) -> Session:
    path = directory / f"{session_id}.json"
    return Session.from_json(path.read_text(encoding="utf-8"))


def list_sessions(directory: Path = SESSIONS_DIR) -> list[dict]:
    rows = []
    for path in sorted(directory.glob("*.json")):
        data = json.loads(path.read_text(encoding="utf-8"))
        rows.append(
            {
                "id": data["session_id"][:8],
                "model": data["model"],
                "turns": len(data["messages"]),
                "updated_at": data["updated_at"],
            }
        )
    return rows

In [5]:
path = save_session(session)
print(f"Wrote {path}")

resumed = load_session(session.session_id)
assert resumed.messages == session.messages
assert resumed.session_id == session.session_id

print(f"Resumed session has {len(resumed.messages)} messages")
print(f"Last turn ({resumed.messages[-1]['role']}):")
print(resumed.messages[-1]["content"][:120])

Wrote _sessions/19dca882-30d8-4491-bca2-5ed7593514a8.json
Resumed session has 2 messages
Last turn (assistant):
Sure! Here's a 3-act outline: (1) arrival, (2) the light fails, (3) what they find below.


## 3. Branching: let users explore alternate paths

A common product need: the user is 15 turns deep and wants to try a different direction *without* losing the main thread. Forking is a one-line operation once your session is plain data — deep-copy it, assign a new id, and record the parent.

In [6]:
def fork_session(
    session: Session,
    up_to_turn: int | None = None,
) -> Session:
    """Return a new session branched from ``session``.

    ``up_to_turn`` truncates messages to the first N entries (checkpoint).
    If None, forks from the current tip.
    """
    forked = deepcopy(session)
    forked.session_id = str(uuid.uuid4())
    forked.parent_id = session.session_id
    forked.created_at = datetime.now(UTC).isoformat()
    forked.updated_at = forked.created_at
    if up_to_turn is not None:
        forked.messages = forked.messages[:up_to_turn]
    return forked

In [7]:
# Take the story outline session and branch from after the first user turn.
branch_a = fork_session(session, up_to_turn=1)
branch_a.append(
    "assistant",
    "Let's try a horror angle: the lighthouse is haunted by previous keepers.",
)

branch_b = fork_session(session, up_to_turn=1)
branch_b.append(
    "assistant",
    "Let's try a literary angle: the keeper is writing letters to a lost friend.",
)

for b in (branch_a, branch_b):
    save_session(b)

print("Sessions on disk:")
for row in list_sessions():
    print(f"  {row['id']}  turns={row['turns']}  updated={row['updated_at']}")

Sessions on disk:
  0fce87d4  turns=2  updated=2026-04-24T00:46:46.498394+00:00
  19dca882  turns=2  updated=2026-04-24T00:46:46.479466+00:00
  471a5100  turns=2  updated=2026-04-24T00:46:46.499379+00:00


Note that both branches share the same `parent_id`. That's enough structure to render a branch picker in a UI or to walk the history when debugging.

## 4. Redacting sensitive content before you save

Never persist what you don't have to. A small pre-save hook is usually enough: strip obvious secrets and give callers the option to drop any turn entirely. This is framework-free on purpose — a real app will plug in its own redactor (regexes, a PII classifier, or a policy engine).

In [8]:
import re

SECRET_PATTERNS = [
    re.compile(r"sk-ant-[a-zA-Z0-9_-]{20,}"),  # Anthropic-style keys
    re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),  # emails
]


def redact(text: str) -> str:
    for pattern in SECRET_PATTERNS:
        text = pattern.sub("[REDACTED]", text)
    return text


def save_session_redacted(session: Session, directory: Path = SESSIONS_DIR) -> Path:
    clean = deepcopy(session)
    for msg in clean.messages:
        if isinstance(msg["content"], str):
            msg["content"] = redact(msg["content"])
    if clean.session_memory:
        clean.session_memory = redact(clean.session_memory)
    return save_session(clean, directory)


leaky = Session.new(model="claude-sonnet-4-6")
leaky.append(
    "user",
    "My key is "
    + "sk-ant-"
    + "api03-abcdefghijklmnop1234567890"
    + " and email me at alice@example.com",
)
leaky.append("assistant", "Understood. I won't share those.")

path = save_session_redacted(leaky)
print(load_session(leaky.session_id).messages[0]["content"])

My key is [REDACTED] and email me at [REDACTED]


## 5. Plugging in a real Claude client (and keeping cache hits on resume)

Everything above is persistence plumbing — no API calls needed. When you're ready to drive a real conversation, the flow is:

```python
import anthropic
client = anthropic.Anthropic()

session = load_session(session_id)  # or Session.new(...)
session.append("user", user_input)

response = client.messages.create(
    model=session.model,
    max_tokens=1024,
    system=session.system,
    messages=session.messages,
)
session.append("assistant", response.content[0].text)
save_session(session)
```

### Preserving prompt-cache hits across resumes

Prompt caching has a ~5-minute idle TTL, so a session reloaded hours later won't hit the cache — that's expected. But *within* a live window, you'll want to keep the cache warm. The rule is: the message prefix must be byte-identical between calls, including where you put `cache_control`. If you always place the breakpoint on the **last user message**, the structure is stable across turns and resume is free.

The `session_memory_compaction` cookbook has a helper `add_cache_control()` that does exactly this; it drops in unchanged here:

```python
messages = add_cache_control(session.messages)  # from the compaction cookbook
response = client.messages.create(..., messages=messages)
```

See [Prompt Caching](prompt_caching.ipynb) for the underlying mechanics.

In [9]:
# Tidy up the demo directory so re-running the notebook starts clean.
for p in SESSIONS_DIR.glob("*.json"):
    p.unlink()
SESSIONS_DIR.rmdir()
print("Cleaned up demo sessions.")

Cleaned up demo sessions.


## Recap

- A session is **plain JSON**: messages, optional session memory, metadata, schema version. Keep it that way and everything downstream (DB, object store, another language) becomes trivial.
- **Save atomically** (write-then-rename) so a crash mid-save can't corrupt a session.
- **Fork by deep-copy + new id**, recording `parent_id`. That's all the structure you need to offer branching in a UI.
- **Redact before persisting.** Treat the on-disk copy as if it might leak.
- **Preserve cache hits on resume** by keeping the message prefix byte-stable — the `add_cache_control` helper from the compaction cookbook is enough.

### Next steps

- **Scale out:** swap `Path.write_text` for S3 / Postgres / your store of choice; the `save_session` / `load_session` interface is the only seam that needs to change.
- **Add compaction:** combine these patterns with the `InstantCompactingChatSession` from [Session Memory Compaction](session_memory_compaction.ipynb) — the session memory is already part of the dataclass.
- **Add cross-session facts** with the [memory tool](../tool_use/memory_cookbook.ipynb) when you want Claude to remember things *between* different sessions, not just within one.